# Collaborative Filtering Evaluation — MovieLens 100k

This notebook evaluates the item-based CF system using the MovieLens 100k dataset.

**Data mapping:**

| MovieLens field | System field | Notes |
|---|---|---|
| `user_id` | `user_id` | direct |
| `item_id` (movie) | `product_id` | movies treated as products |
| `rating` (1–5) | `score` | explicit ratings used as interaction weights |

**Evaluation strategy:** 5-fold cross-validation (u1–u5 splits)

**Metrics:** Precision@K, Recall@K, NDCG@K, Hit-Rate@K

**Relevance:** items rated ≥ 4 are considered "liked"

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import app modules
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "ml-100k"
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Data exists  : {DATA_DIR.exists()}")

In [ ]:
import logging
import time
import warnings
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

from app.evaluation.cf_evaluator import (
    FOLD_PAIRS,
    RELEVANCE_THRESHOLD,
    MetricResult,
    build_user_item_matrix,
    compute_item_similarity,
    compute_metrics_at_k,
    evaluate_fold,
    generate_user_recommendations,
    load_ml100k_split,
    run_5fold_cv,
)

print("All imports OK")

## 1. Dataset overview

In [ ]:
cols = ["user_id", "product_id", "score", "timestamp"]
full_df = pd.read_csv(DATA_DIR / "u.data", sep="\t", names=cols)

print(f"Total ratings : {len(full_df):,}")
print(f"Unique users  : {full_df['user_id'].nunique():,}")
print(f"Unique movies : {full_df['product_id'].nunique():,}")
print(f"Rating range  : {full_df['score'].min()} – {full_df['score'].max()}")
print(f"Sparsity      : {1 - len(full_df) / (full_df['user_id'].nunique() * full_df['product_id'].nunique()):.2%}")
full_df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
ax = axes[0]
full_df["score"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Rating Distribution", fontsize=13)
ax.set_xlabel("Rating")
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Ratings per user (log scale)
ax = axes[1]
user_counts = full_df["user_id"].value_counts()
ax.hist(user_counts, bins=40, color="coral", edgecolor="white")
ax.set_title("Ratings per User", fontsize=13)
ax.set_xlabel("# Ratings")
ax.set_ylabel("# Users")

# Ratings per movie (log scale)
ax = axes[2]
item_counts = full_df["product_id"].value_counts()
ax.hist(item_counts, bins=40, color="seagreen", edgecolor="white")
ax.set_title("Ratings per Movie", fontsize=13)
ax.set_xlabel("# Ratings")
ax.set_ylabel("# Movies")

fig.tight_layout()
plt.suptitle("MovieLens 100k — Data Overview", y=1.02, fontsize=14, fontweight="bold")
plt.show()

## 2. Sanity check — single fold

In [ ]:
base_df, test_df = load_ml100k_split(DATA_DIR / "u1.base", DATA_DIR / "u1.test")
print(f"Train: {len(base_df):,} interactions | Test: {len(test_df):,} interactions")
print(f"Train users: {base_df['user_id'].nunique()} | Train movies: {base_df['product_id'].nunique()}")
print(f"Test  users: {test_df['user_id'].nunique()}  | Test  movies: {test_df['product_id'].nunique()}")

relevant_in_test = test_df[test_df["score"] >= RELEVANCE_THRESHOLD]
print(f"\nRelevant test items (rating >= {RELEVANCE_THRESHOLD}): {len(relevant_in_test):,} ({len(relevant_in_test)/len(test_df):.1%})")

In [ ]:
TOP_K = 20
K_VALUES = (5, 10, 20)

print("Building user-item matrix...")
t0 = time.perf_counter()
sparse_mat, user_ids, product_ids = build_user_item_matrix(base_df)
print(f"Matrix shape: {sparse_mat.shape}  nnz={sparse_mat.nnz:,}  ({time.perf_counter()-t0:.1f}s)")

print("\nComputing item similarities...")
t1 = time.perf_counter()
item_similarities = compute_item_similarity(sparse_mat, product_ids, top_k=TOP_K)
print(f"Items with similarities: {len(item_similarities):,} / {len(product_ids)}  ({time.perf_counter()-t1:.1f}s)")

In [ ]:
# Example: most similar movies for movie #1
example_movie = "1"
movie_info = pd.read_csv(DATA_DIR / "u.item", sep="|", encoding="latin-1",
                         names=["id", "title"] + [f"f{i}" for i in range(22)],
                         usecols=["id", "title"])
id_to_title = dict(zip(movie_info["id"].astype(str), movie_info["title"]))

print(f"Seed movie : {id_to_title.get(example_movie, '?')}")
print("\nTop-10 similar movies:")
for i, (mid, score) in enumerate(item_similarities.get(example_movie, [])[:10], 1):
    print(f"  {i:2}. [{mid:>4}] {id_to_title.get(mid, '?'):<50} score={score:.4f}")

## 3. Full 5-fold cross-validation

In [ ]:
t_total = time.perf_counter()

averaged = run_5fold_cv(
    data_dir=DATA_DIR,
    top_k=TOP_K,
    k_values=K_VALUES,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

print(f"\nTotal time: {time.perf_counter() - t_total:.1f}s")

In [ ]:
results_df = pd.DataFrame(
    [
        {
            "K": k,
            "Precision@K": r.precision,
            "Recall@K": r.recall,
            "NDCG@K": r.ndcg,
            "HitRate@K": r.hit_rate,
            "Avg Users": r.n_users,
        }
        for k, r in sorted(averaged.items())
    ]
).set_index("K")

results_df.style \
    .format({"Precision@K": "{:.4f}", "Recall@K": "{:.4f}", "NDCG@K": "{:.4f}", "HitRate@K": "{:.4f}"}) \
    .background_gradient(cmap="YlGn", subset=["Precision@K", "Recall@K", "NDCG@K", "HitRate@K"]) \
    .set_caption("5-Fold CV Results — Item-based CF (MovieLens 100k)")

## 4. Results visualization

In [ ]:
metrics = ["Precision@K", "Recall@K", "NDCG@K", "HitRate@K"]
colors  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)

for ax, metric, color in zip(axes, metrics, colors):
    ks = sorted(averaged.keys())
    vals = [getattr(averaged[k], metric.replace("@K", "").lower().replace("-", "_")) for k in ks]
    # map metric name to MetricResult field
    field_map = {
        "Precision@K": "precision",
        "Recall@K": "recall",
        "NDCG@K": "ndcg",
        "HitRate@K": "hit_rate",
    }
    vals = [getattr(averaged[k], field_map[metric]) for k in ks]

    bars = ax.bar([str(k) for k in ks], vals, color=color, alpha=0.85, edgecolor="white", width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.3f}", ha="center", va="bottom", fontsize=10)

    ax.set_title(metric, fontsize=13, fontweight="bold")
    ax.set_xlabel("K")
    ax.set_ylim(0, min(1.0, max(vals) * 1.3 + 0.05))
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    f"Item-based CF on MovieLens 100k\n5-Fold Cross-Validation  |  top_k={TOP_K}  |  relevance≥{RELEVANCE_THRESHOLD}",
    fontsize=13, y=1.04,
)
fig.tight_layout()
plt.show()

In [ ]:
# Line chart — metrics vs K
fig, ax = plt.subplots(figsize=(8, 5))

ks = sorted(averaged.keys())
for metric, field, color, marker in [
    ("Precision@K", "precision", "#4C72B0", "o"),
    ("Recall@K",    "recall",    "#DD8452", "s"),
    ("NDCG@K",      "ndcg",      "#55A868", "^"),
    ("HitRate@K",   "hit_rate",  "#C44E52", "D"),
]:
    vals = [getattr(averaged[k], field) for k in ks]
    ax.plot([str(k) for k in ks], vals, marker=marker, label=metric, color=color, linewidth=2, markersize=8)

ax.set_xlabel("K", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("CF Evaluation Metrics vs. K  (5-fold average)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Sensitivity analysis — effect of TOP_K on quality

In [ ]:
# Compare different TOP_K values (neighbors stored per item)
# This shows how many similar-item neighbors are needed to get good coverage

topk_sweep = [10, 20, 50]
eval_k = 10  # fix evaluation cutoff to K=10

sweep_results = {}
for tk in topk_sweep:
    print(f"Running sweep: top_k={tk} ...")
    r = run_5fold_cv(data_dir=DATA_DIR, top_k=tk, k_values=(eval_k,))
    sweep_results[tk] = r[eval_k]
    print(f"  @K={eval_k}: P={r[eval_k].precision:.4f}  R={r[eval_k].recall:.4f}  NDCG={r[eval_k].ndcg:.4f}  HR={r[eval_k].hit_rate:.4f}")

In [ ]:
sweep_df = pd.DataFrame(
    [{"top_k": tk, **{"Precision": r.precision, "Recall": r.recall, "NDCG": r.ndcg, "HitRate": r.hit_rate}}
     for tk, r in sweep_results.items()]
).set_index("top_k")

fig, ax = plt.subplots(figsize=(8, 4))
sweep_df.plot(marker="o", ax=ax, linewidth=2, markersize=8)
ax.set_xlabel("top_k (neighbors stored per item)", fontsize=12)
ax.set_ylabel("Score @ K=10", fontsize=12)
ax.set_title(f"Effect of top_k on CF quality  (5-fold avg, eval K={eval_k})", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

sweep_df.style.format("{:.4f}").background_gradient(cmap="YlGn")

## 6. Sensitivity analysis — effect of relevance threshold

In [ ]:
threshold_sweep = [3, 4, 5]
eval_k = 10

threshold_results = {}
for thr in threshold_sweep:
    print(f"Relevance threshold = {thr} ...")
    r = run_5fold_cv(data_dir=DATA_DIR, top_k=TOP_K, k_values=(eval_k,), relevance_threshold=thr)
    threshold_results[thr] = r[eval_k]
    print(f"  P={r[eval_k].precision:.4f}  R={r[eval_k].recall:.4f}  NDCG={r[eval_k].ndcg:.4f}  HR={r[eval_k].hit_rate:.4f}")

thr_df = pd.DataFrame(
    [{"threshold": thr, "Precision": r.precision, "Recall": r.recall, "NDCG": r.ndcg, "HitRate": r.hit_rate}
     for thr, r in threshold_results.items()]
).set_index("threshold")

thr_df.style.format("{:.4f}").background_gradient(cmap="YlGn")

## 7. Summary & interpretation

### What the metrics mean

| Metric | Meaning |
|---|---|
| **Precision@K** | Of K recommended movies, what fraction did the user actually like (≥ threshold)? |
| **Recall@K** | Of all movies the user liked in the test set, what fraction appear in the top-K? |
| **NDCG@K** | Are higher-rated movies ranked earlier? (1.0 = perfect order, 0.0 = worst) |
| **Hit-Rate@K** | For what fraction of users did at least one relevant movie appear in top-K? |

### Known limitations of this evaluation

1. **Only positive test items count** — items rated below threshold in the test set are neither penalized nor rewarded.
2. **No coverage metric** — some items/users may never appear in any recommendation.
3. **Cold-start not tested** — all test users have training data; truly new users would score 0.
4. **Timestamp ordering not preserved** — splits are random, so future-only evaluation is not enforced.